# Multimodal Fusion: Additive (Element-wise Sum) Baseline

This notebook implements **Additive Fusion** for in-hospital mortality prediction.
- **Temporal Sequence Features**: $t \in \mathbb{R}^{64}$
- **Static Tabular Features**: $s \in \mathbb{R}^{64}$
- **Additive Formulation**: Both modalities are standardized independently, then combined via element-wise addition:
  $$X_{\text{fused}} = t_{\text{norm}} + s_{\text{norm}} \in \mathbb{R}^{64}$$


## Step 1: Imports, Configuration & Device Setup
Load dependencies, set seed (42), and detect compute device (MPS / CUDA / CPU).


In [ ]:
from pathlib import Path
import copy
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, accuracy_score,
    balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 50
PATIENCE = 7
LR = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_start, *_start.parents] if (p / "checkpoints").exists() and (p / "processed").exists()), _start)
FUSION_DIR = PROJECT_ROOT / "checkpoints" / "fusion"
OUT_DIR = FUSION_DIR / "additive_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Output:", OUT_DIR.resolve())




## Step 2: Load Multimodal Embeddings & Validate Splits
Load temporal and tabular embeddings from `fused_embeddings_{split}.npz` and verify split isolation.


In [ ]:
def load_split(split):
    path = FUSION_DIR / f"fused_embeddings_{split}.npz"

    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path) as data:
        temporal = data["temporal_embeddings"].astype(np.float32)
        tabular = data["tabular_embeddings"].astype(np.float32)
        ids = data["patient_ids"].reshape(-1).astype(np.int64)
        labels = data["labels"].reshape(-1).astype(np.float32)

    assert temporal.shape == tabular.shape
    assert temporal.ndim == 2 and temporal.shape[1] == 64
    assert len(temporal) == len(ids) == len(labels)
    assert len(np.unique(ids)) == len(ids)
    assert np.isfinite(temporal).all()
    assert np.isfinite(tabular).all()
    assert np.isfinite(labels).all()
    assert set(np.unique(labels)).issubset({0.0, 1.0})

    return temporal, tabular, labels, ids


t_train, s_train, y_train, ids_train = load_split("train")
t_val, s_val, y_val, ids_val = load_split("val")
t_test, s_test, y_test, ids_test = load_split("test")

assert not (set(ids_train) & set(ids_val))
assert not (set(ids_train) & set(ids_test))
assert not (set(ids_val) & set(ids_test))

print("Train:", t_train.shape)
print("Validation:", t_val.shape)
print("Test:", t_test.shape)



## Step 3: Modality Standardization & Additive Fusion
Normalize each modality using training set statistics and compute element-wise sum $X = t + s \in \mathbb{R}^{64}$.


In [ ]:
# Standardize each modality using training data only.
t_mean, t_std = t_train.mean(axis=0), t_train.std(axis=0)
s_mean, s_std = s_train.mean(axis=0), s_train.std(axis=0)

t_std[t_std < 1e-8] = 1.0
s_std[s_std < 1e-8] = 1.0

t_train = (t_train - t_mean) / t_std
t_val = (t_val - t_mean) / t_std
t_test = (t_test - t_mean) / t_std

s_train = (s_train - s_mean) / s_std
s_val = (s_val - s_mean) / s_std
s_test = (s_test - s_mean) / s_std

# Add the corresponding 64 dimensions from both modalities.
X_train = (t_train + s_train).astype(np.float32)
X_val = (t_val + s_val).astype(np.float32)
X_test = (t_test + s_test).astype(np.float32)

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == 64
assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

np.savez(
    OUT_DIR / "additive_scaler.npz",
    temporal_mean=t_mean,
    temporal_std=t_std,
    tabular_mean=s_mean,
    tabular_std=s_std
)




## Step 4: PyTorch DataLoaders & Class Weighting
Construct batch loaders for 64-dimensional additive vectors and compute positive class loss weight.


In [ ]:
def make_loader(X, y, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(y.astype(np.float32))
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0
    )


train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val, False)
test_loader = make_loader(X_test, y_test, False)




## Step 5: Downstream Classifier Architecture
Define 2-layer MLP classifier ($64 \to 64 \to 1$) with ReLU and 25% dropout, loss criterion, optimizer, and prediction helper.


In [ ]:
class FusionClassifier(nn.Module):
    def __init__(self, input_dim=64, dropout=0.25):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


model = FusionClassifier().to(device)

n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        n_neg / max(n_pos, 1),
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

print(f"Training positives: {n_pos:,}")
print(f"Training negatives: {n_neg:,}")


def predict_probabilities(loader):
    model.eval()
    labels, probabilities = [], []

    with torch.no_grad():
        for xb, yb in loader:
            probs = torch.sigmoid(model(xb.to(device)))
            labels.append(yb.numpy())
            probabilities.append(probs.cpu().numpy())

    return np.concatenate(labels), np.concatenate(probabilities)




## Step 6: Training Loop with Validation AUPRC Early Stopping
Train for up to 50 epochs with patience = 7. Save best checkpoint to `additive_classifier.pt`.


In [ ]:
best_val_ap = -np.inf
best_state = None
wait = 0
history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(yb)
        total_samples += len(yb)

    train_loss = total_loss / total_samples
    yv, pv = predict_probabilities(val_loader)

    val_ap = average_precision_score(yv, pv)
    val_auc = roc_auc_score(yv, pv)

    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_auprc": float(val_ap),
        "val_auroc": float(val_auc)
    })

    print(
        f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Val AUPRC={val_ap:.4f} | "
        f"Val AUROC={val_auc:.4f}"
    )

    if np.isfinite(val_ap) and val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("No valid checkpoint selected.")

model.load_state_dict(best_state)

torch.save(model.state_dict(), OUT_DIR / "additive_classifier.pt")

with open(OUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"Best validation AUPRC: {best_val_ap:.4f}")



## Step 7: Decision Threshold Tuning (Validation F1)
Find the classification threshold that maximizes F1 score on the validation set.


In [ ]:
# Select the classification threshold using validation F1 only.
yv, pv = predict_probabilities(val_loader)
p_curve, r_curve, thresholds = precision_recall_curve(yv, pv)

f1_curve = (
    2 * p_curve[:-1] * r_curve[:-1]
    / (p_curve[:-1] + r_curve[:-1] + 1e-12)
)

best_idx = int(np.argmax(f1_curve))
best_threshold = float(thresholds[best_idx])

print(f"Selected threshold: {best_threshold:.4f}")
print(f"Validation F1: {f1_curve[best_idx]:.4f}")



## Step 8: Held-Out Test Evaluation & Metrics Export
Evaluate the test set once using the tuned threshold and save comprehensive metrics to `test_metrics.json`.


In [ ]:
# Evaluate the held-out test set using the selected threshold.
yt, pt = predict_probabilities(test_loader)
yt_pred = (pt >= best_threshold).astype(int)

results = {
    "fusion": "additive",
    "classifier": "MLP",
    "threshold": best_threshold,
    "test_auroc": float(roc_auc_score(yt, pt)),
    "test_auprc": float(average_precision_score(yt, pt)),
    "test_accuracy": float(accuracy_score(yt, yt_pred)),
    "test_balanced_accuracy": float(
        balanced_accuracy_score(yt, yt_pred)
    ),
    "test_precision": float(
        precision_score(yt, yt_pred, zero_division=0)
    ),
    "test_recall": float(
        recall_score(yt, yt_pred, zero_division=0)
    ),
    "test_f1": float(f1_score(yt, yt_pred, zero_division=0)),
    "confusion_matrix": confusion_matrix(
        yt, yt_pred, labels=[0, 1]
    ).tolist()
}

print("\nADDITIVE FUSION RESULTS")
for key, value in results.items():
    print(f"{key}: {value}")

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved results to:", OUT_DIR.resolve())